# 안전귀가Navi — Day 3 주소→좌표 + 반경 검색

**전제:** `.env` 파일에 `KAKAO_REST_API_KEY=...` 설정되어 있어야 함.

## 검증 항목
1. 카카오맵 API 호출 정상 동작 (도로명·지번·키워드)
2. `get_nearby_facilities` 반경별 카운트
3. `nearest_per_type` type별 최근접 거리
4. `nearest_safe_route` 안심귀갓길 거리
5. `analyze_address` 통합 진입점 (Day 4 점수 엔진 입력 형식)

In [ ]:
import sys
from pathlib import Path
import json

sys.path.insert(0, str(Path(r'C:\Users\pjhic\Projects\Seoul_bigdata\src')))

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import search
print(f'API 키 설정: {bool(search.KAKAO_KEY)}')
print(f'기본 반경: {search.DEFAULT_RADII}')
print(f'점수 항목 type: {search.SCORING_TYPES}')

## 1. Geocoding 동작 확인

In [ ]:
test_addresses = [
    '서울특별시 관악구 봉천로 100',         # 도로명
    '서울특별시 마포구 양화로 45',           # 도로명
    '서울특별시 종로구 자하문로17길 36',      # 1인 가구 자취 자주 있는 지역
    '강남역',                                # 키워드 fallback
    '서울특별시 송파구 잠실동',              # 자치구 동 단위
]

for addr in test_addresses:
    try:
        lat, lon = search.geocode(addr)
        print(f'  {addr:<40} → ({lat:.4f}, {lon:.4f})')
    except Exception as e:
        print(f'  {addr:<40} → ERROR: {e}')

## 2. 반경 검색 (50/100/300/500m)

In [ ]:
lat, lon = search.geocode('서울특별시 관악구 봉천로 100')
result = search.get_nearby_facilities(lat, lon)

print(f"좌표: ({lat:.4f}, {lon:.4f})\n")
print(f"{'반경':>6}  {'합계':>4}  {'cctv':>4}  {'light':>4}  {'bell':>4}  {'facility':>4}")
for r, info in result['radii'].items():
    by_type = info['by_type']
    print(f"{r:>5}m  {info['total']:>4}  {by_type['cctv']:>4}  {by_type['light']:>4}  {by_type['bell']:>4}  {by_type['facility']:>4}")

## 3. type별 최근접 시설물 거리
데이터가 적은 자치구(송파구 등)도 0점이 안 되도록 거리 기반 신호 활용.

In [ ]:
samples = ['서울특별시 관악구 봉천로 100', '서울특별시 송파구 잠실동', '서울특별시 종로구 자하문로17길 36']

for addr in samples:
    lat, lon = search.geocode(addr)
    nearest = search.nearest_per_type(lat, lon)
    print(f'\n[{addr}]')
    for t, dist in nearest.items():
        if dist is None:
            print(f'  {t:8s}: 데이터 없음')
        else:
            print(f'  {t:8s}: {dist:>7.1f}m')

## 4. 가장 가까운 안심귀갓길

In [ ]:
for addr in samples:
    lat, lon = search.geocode(addr)
    route = search.nearest_safe_route(lat, lon)
    print(f'\n[{addr}]')
    print(f'  가장 가까운 안심귀갓길: {route["route_name"]} ({route["gu"]})')
    print(f'  거리: {route["distance_m"]:.1f}m')
    print(f'  300m 내 안심귀갓길: {route["within_300m_count"]}개 / 500m 내: {route["within_500m_count"]}개')

## 5. analyze_address 통합 진입점
Day 4 점수 엔진이 받게 될 입력 형식 확인.

In [ ]:
data = search.analyze_address('서울특별시 관악구 봉천로 100')
print(json.dumps(data, indent=2, ensure_ascii=False))

## 6. 실행 시간 측정
Day 4·5에서 사용자가 주소 입력 시 응답 속도 예측.

In [ ]:
import time

addr = '서울특별시 관악구 봉천로 100'

t0 = time.perf_counter()
lat, lon = search.geocode(addr)
t_geocode = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
search.get_nearby_facilities(lat, lon)
t_nearby = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
search.nearest_per_type(lat, lon)
t_nearest = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
search.nearest_safe_route(lat, lon)
t_route = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
search.analyze_address(addr)
t_analyze = (time.perf_counter() - t0) * 1000

print(f'geocode (네트워크 호출):     {t_geocode:>7.2f} ms')
print(f'get_nearby_facilities:      {t_nearby:>7.2f} ms')
print(f'nearest_per_type:           {t_nearest:>7.2f} ms')
print(f'nearest_safe_route:         {t_route:>7.2f} ms')
print(f'─' * 45)
print(f'analyze_address (통합):     {t_analyze:>7.2f} ms')